# Curadoria — Pipeline FALSO (Google Fact Check)

Lê todos os CSVs raw do pipeline `pipeline_falso_google_factcheck/raw/`,
acumula o histórico, aplica limpeza e padronização e salva um CSV curated com timestamp.

**Regras desta camada:**
- Remover HTML do `texto_afirmacao`
- Normalizar espaços e capitalização
- Padronizar datas para `YYYY-MM-DD`
- Remover duplicatas por `texto_principal + fonte + url_origem`
- Mapear `avaliacao_original` para categoria normalizada
- Adicionar colunas do schema obrigatório curated
- **Não** modificar arquivos raw
- **Não** gerar dataset final de treino

**v2 — Correções nesta versão:**
- Mapa de avaliações expandido (+12 entradas)
- Normalização unicode (NFKC) antes do lookup
- Fallback ASCII para encoding corrompido (`não_é_bem_assim`)
- Fallback por prefixo para ratings em forma de sentença
- `confere` adicionado como VERDADEIRO


## Bibliotecas

In [13]:
import re
import unicodedata
import uuid
import pandas as pd

from datetime import datetime
from pathlib import Path
from html.parser import HTMLParser

## Configuração de caminhos

In [14]:
NOME_PIPELINE = "pipeline_falso_google_factcheck"

PASTA_RAW     = Path(f"../dados/{NOME_PIPELINE}/raw")
PASTA_CURATED = Path(f"../dados/{NOME_PIPELINE}/curated")
PASTA_CURATED.mkdir(parents=True, exist_ok=True)

print(f"Raw:     {PASTA_RAW}")
print(f"Curated: {PASTA_CURATED}")

Raw:     ..\dados\pipeline_falso_google_factcheck\raw
Curated: ..\dados\pipeline_falso_google_factcheck\curated


## Funções utilitárias

In [15]:
class _StripHTML(HTMLParser):
    """Parser simples que descarta tags e acumula apenas o texto."""
    def __init__(self):
        super().__init__()
        self._partes = []

    def handle_data(self, data):
        self._partes.append(data)

    def get_text(self):
        return " ".join(self._partes)


def remover_html(texto: str) -> str:
    """Remove tags HTML e normaliza espaços em branco."""
    if not isinstance(texto, str) or not texto.strip():
        return ""
    parser = _StripHTML()
    parser.feed(texto)
    limpo = parser.get_text()
    limpo = re.sub(r"\s+", " ", limpo).strip()
    return limpo


def padronizar_data(valor) -> str:
    """
    Tenta converter datas em vários formatos para YYYY-MM-DD.
    Retorna string vazia se não conseguir.
    """
    if not isinstance(valor, str) or not valor.strip():
        return ""
    valor = valor.strip()
    formatos = [
        "%Y-%m-%dT%H:%M:%SZ",
        "%Y-%m-%dT%H:%M:%S",
        "%Y-%m-%d",
        "%d/%m/%Y",
        "%d/%m/%Y %H:%M:%S",
    ]
    for fmt in formatos:
        try:
            return datetime.strptime(valor, fmt).strftime("%Y-%m-%d")
        except ValueError:
            continue
    return ""


# ===========================================================================
# MAPA DE AVALIAÇÕES — v2 (corrigido e expandido)
# Expandido para cobrir valores antes classificados incorretamente como OUTRO.
# Chaves sempre em minúsculo — a função normaliza antes de consultar.
# ===========================================================================
_MAPA_AVALIACAO = {
    # --- FALSO ---
    "falso":                   "FALSO",
    "false":                   "FALSO",
    "incorreto":               "FALSO",
    "incorrect":               "FALSO",
    "mentira":                 "FALSO",
    "errado":                  "FALSO",           # novo: 107 ocorrências
    "insustentável":           "FALSO",           # novo: 37 ocorrências
    "insustentavel":           "FALSO",           # novo: variante sem acento
    "montagem":                "FALSO",           # novo: imagem manipulada (9x)
    "sátira":                  "FALSO",           # novo: usada para disseminar falso (7x)
    "satira":                  "FALSO",           # novo: variante sem acento
    "boato":                   "FALSO",           # novo: 1 ocorrência
    "predominantemente falso": "FALSO",           # novo: 1 ocorrência
    # --- ENGANOSO ---
    "enganoso":                "ENGANOSO",
    "misleading":              "ENGANOSO",
    "distorcido":              "ENGANOSO",
    "parcialmente falso":      "ENGANOSO",
    "mostly false":            "ENGANOSO",
    "half true":               "ENGANOSO",
    "não_é_bem_assim":         "ENGANOSO",        # novo: Boatos.org (91x)
    "nao_e_bem_assim":         "ENGANOSO",        # novo: fallback ASCII de encoding
    "não é bem assim":         "ENGANOSO",        # novo: variante com espaços
    "nao e bem assim":         "ENGANOSO",        # novo: variante ASCII com espaços
    "enganador":               "ENGANOSO",        # novo: 15-17 ocorrências
    # --- FORA_DE_CONTEXTO ---
    "fora de contexto":        "FORA_DE_CONTEXTO",
    "sem contexto":            "FORA_DE_CONTEXTO",
    "out of context":          "FORA_DE_CONTEXTO",
    "falta contexto":          "FORA_DE_CONTEXTO", # novo: 4 ocorrências
    # --- IMPRECISO ---
    "impreciso":               "IMPRECISO",
    "exagerado":               "IMPRECISO",
    # --- NAO_VERIFICAVEL ---
    "não verificável":         "NAO_VERIFICAVEL",
    "nao verificavel":         "NAO_VERIFICAVEL",  # variante sem acento
    "unverified":              "NAO_VERIFICAVEL",
    # --- VERDADEIRO ---
    "verdadeiro":              "VERDADEIRO",
    "true":                    "VERDADEIRO",
    "comprovado":              "VERDADEIRO",
    "certo":                   "VERDADEIRO",
    "correto":                 "VERDADEIRO",
    "fato":                    "VERDADEIRO",
    "fato verificado":         "VERDADEIRO",
    "confirmado":              "VERDADEIRO",
    "confere":                 "VERDADEIRO",       # novo: ausente no mapa anterior
}


def _normalizar_chave(texto: str) -> str:
    """
    Normaliza texto para lookup no mapa:
    1. NFKC — resolve compatibilidade unicode
    2. Remove U+FFFD gerado por encoding corrompido nos arquivos raw
    3. Colapsa underscores múltiplos (artefato da remoção de chars)
    4. Colapsa espaços múltiplos
    5. Strip + lower
    """
    if not isinstance(texto, str):
        return ""
    texto = unicodedata.normalize("NFKC", texto)
    texto = texto.replace("\ufffd", "")
    texto = re.sub(r"_+", "_", texto)
    texto = re.sub(r"\s+", " ", texto)
    return texto.strip().lower()


_TRANSLITERACAO = str.maketrans(
    "ãáâàéêèíîóôõúûçÃÁÂÀÉÊÈÍÎÓÔÕÚÛÇ",
    "aaaaeeeiiooouucAAAAEEEIIOOOUUC",
)


def _ascii_simples(texto: str) -> str:
    """Transliteração PT→ASCII para fallback de encoding corrompido."""
    return texto.translate(_TRANSLITERACAO)


# Prefixos que identificam o veredicto mesmo em ratings de forma de sentença.
# Ex.: 'Falso: O vídeo mostra...' → prefixo 'falso:' → FALSO
_PREFIXOS_CATEGORIA = [
    ("falso:",                "FALSO"),
    ("falso -",               "FALSO"),
    ("é falso",               "FALSO"),
    ("e falso",               "FALSO"),
    ("sao falsas",            "FALSO"),
    ("sao falsos",            "FALSO"),
    ("enganoso:",             "ENGANOSO"),
    ("enganoso -",            "ENGANOSO"),
    ("é enganoso",            "ENGANOSO"),
    ("e enganoso",            "ENGANOSO"),
    ("enganosa",              "ENGANOSO"),
    ("verdadeiro:",           "VERDADEIRO"),
    ("verdadeiro -",          "VERDADEIRO"),
    ("comprovado:",           "VERDADEIRO"),
    ("comprovado -",          "VERDADEIRO"),
    ("sem contexto:",         "FORA_DE_CONTEXTO"),
    ("falta contexto:",       "FORA_DE_CONTEXTO"),
    ("esta fora de contexto", "FORA_DE_CONTEXTO"),
    ("contextualizando:",     "FORA_DE_CONTEXTO"),
]


def _mapear_por_prefixo(chave: str) -> str | None:
    """
    Tenta mapear ratings em forma de sentença a partir do prefixo.
    Retorna None se nenhum prefixo casar.
    """
    for prefixo, categoria in _PREFIXOS_CATEGORIA:
        if chave.startswith(prefixo):
            return categoria
    return None


def mapear_avaliacao(valor: str) -> str:
    """
    Normaliza a avaliação original para uma categoria padronizada.

    Pipeline de resolução (4 etapas):
    1. Lookup direto no mapa (chave normalizada via unicode NFKC)
    2. Lookup com transliteração ASCII (fallback para encoding corrompido)
    3. Fallback por prefixo — chave normalizada (ratings em sentença)
    4. Fallback por prefixo — chave ASCII
    5. OUTRO para tudo que não casar
    """
    if not isinstance(valor, str) or not valor.strip():
        return "NAO_CLASSIFICADO"

    chave       = _normalizar_chave(valor)
    chave_ascii = _ascii_simples(chave)

    return (
        _MAPA_AVALIACAO.get(chave)
        or _MAPA_AVALIACAO.get(chave_ascii)
        or _mapear_por_prefixo(chave)
        or _mapear_por_prefixo(chave_ascii)
        or "OUTRO"
    )


print("Funções utilitárias v2 definidas.")
print(f"  Entradas no mapa de avaliações : {len(_MAPA_AVALIACAO)}")
print(f"  Prefixos de fallback           : {len(_PREFIXOS_CATEGORIA)}")


Funções utilitárias v2 definidas.
  Entradas no mapa de avaliações : 42
  Prefixos de fallback           : 19


## Leitura de todos os CSVs raw

> Arquivos com `_TESTE_` no nome são ignorados automaticamente — eles são gerados em MODO_TESTE e não devem entrar na curadoria oficial.

In [16]:
todos_csvs   = sorted(PASTA_RAW.glob("*.csv"))
arquivos_raw = [a for a in todos_csvs if "_TESTE_" not in a.name]
ignorados    = [a for a in todos_csvs if "_TESTE_" in a.name]

if ignorados:
    print(f"Arquivos _TESTE_ ignorados ({len(ignorados)}):")
    for arq in ignorados:
        print(f"  (ignorado) {arq.name}")

print(f"\nArquivos raw oficiais encontrados: {len(arquivos_raw)}")
for arq in arquivos_raw:
    print(f"  {arq.name}")

if not arquivos_raw:
    raise FileNotFoundError("Nenhum arquivo raw oficial encontrado em " + str(PASTA_RAW))

frames = []
for arq in arquivos_raw:
    df_arq = pd.read_csv(arq, encoding="utf-8-sig", dtype=str)
    df_arq["arquivo_raw_origem"] = arq.name
    frames.append(df_arq)

df_raw = pd.concat(frames, ignore_index=True)

# Sanitização de segurança: remover url_consulta se presente (pode conter API key)
if "url_consulta" in df_raw.columns:
    df_raw = df_raw.drop(columns=["url_consulta"])
    print("Sanitização: coluna url_consulta removida dos raws (pode conter API key)")

print(f"\nTotal bruto acumulado: {len(df_raw)} registros")
df_raw.head(3)

Arquivos _TESTE_ ignorados (1):
  (ignorado) google_factcheck_raw_TESTE_2026-05-16_21-55-45.csv

Arquivos raw oficiais encontrados: 8
  google_factcheck_raw.csv
  google_factcheck_raw_2026-04-30_00-58-54.csv
  google_factcheck_raw_2026-05-09_15-26-18.csv
  google_factcheck_raw_2026-05-09_15-32-24.csv
  google_factcheck_raw_2026-05-10_02-42-39.csv
  google_factcheck_raw_2026-05-12_23-02-04.csv
  google_factcheck_raw_2026-05-17_02-42-35.csv
  google_factcheck_raw_2026-05-30_21-44-39.csv

Total bruto acumulado: 9818 registros


,termo_busca,texto_afirmacao,data_claim,fonte,url_checagem,avaliacao_original,data_publicacao,arquivo_raw_origem,fonte_verificacao,url_consulta,data_coleta,origem_pipeline,query_matched,tema_query,subtema_query
0,urnas eletrônicas,Lula perdeu todas as eleições com votação manu...,2025-12-07T00:00:00Z,AFP Checamos,https://checamos.afp.com/doc.afp.com.88868T3,Enganoso,2025-12-15T17:22:00Z,google_factcheck_raw.csv,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,urnas eletrônicas,Lula perdeu todas as eleições feitas com cédul...,2025-12-09T00:00:00Z,Aos Fatos,https://www.aosfatos.org/noticias/falso-que-lu...,falso,2025-12-09T00:00:00Z,google_factcheck_raw.csv,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,urnas eletrônicas,No congresso americano todas as urnas foram ha...,2025-11-02T00:00:00Z,Projeto Comprova,https://projetocomprova.com.br/publica%C3%A7%C...,Falso,2025-11-07T00:00:00Z,google_factcheck_raw.csv,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Limpeza e padronização

In [17]:
df = df_raw.copy()

# --- texto_principal ---
df["texto_principal"] = df["texto_afirmacao"].apply(remover_html)

# Descartar registros sem texto
antes = len(df)
df = df[df["texto_principal"].str.strip() != ""].copy()
print(f"Removidos sem texto: {antes - len(df)}")

# --- fonte normalizada ---
df["fonte"] = (
    df["fonte_verificacao"]
    .fillna("DESCONHECIDA")
    .str.strip()
    .str.upper()
    .str.replace(r"\s+", "_", regex=True)
)

# --- datas padronizadas ---
df["data_publicacao"] = df["data_publicacao"].apply(padronizar_data)

# --- avaliação normalizada ---
df["avaliacao_original"] = df["avaliacao_original"].fillna("").str.strip()
df["avaliacao_categoria"] = df["avaliacao_original"].apply(mapear_avaliacao)

# --- url_origem ---
df["url_origem"] = df["url_checagem"].fillna("").str.strip()

print("Limpeza aplicada.")
print(f"\nDistribuição de avaliacao_categoria:")
print(df["avaliacao_categoria"].value_counts())

Removidos sem texto: 0
Limpeza aplicada.

Distribuição de avaliacao_categoria:
avaliacao_categoria
FALSO               6871
ENGANOSO            2535
FORA_DE_CONTEXTO     237
OUTRO                 95
VERDADEIRO            68
IMPRECISO             12
Name: count, dtype: int64


## Diagnóstico — comparação mapeamento anterior vs. corrigido

Quantifica o ganho da correção executando o mapa **anterior** sobre os mesmos dados.
Executado antes da deduplicação para preservar os números brutos.


In [18]:
# Mapa anterior reproduzido para comparação
_MAPA_ANTERIOR = {
    "falso": "FALSO", "false": "FALSO", "incorreto": "FALSO",
    "incorrect": "FALSO", "mentira": "FALSO",
    "enganoso": "ENGANOSO", "misleading": "ENGANOSO", "distorcido": "ENGANOSO",
    "parcialmente falso": "ENGANOSO", "mostly false": "ENGANOSO", "half true": "ENGANOSO",
    "fora de contexto": "FORA_DE_CONTEXTO", "sem contexto": "FORA_DE_CONTEXTO",
    "out of context": "FORA_DE_CONTEXTO",
    "impreciso": "IMPRECISO", "exagerado": "IMPRECISO",
    "não verificável": "NAO_VERIFICAVEL", "unverified": "NAO_VERIFICAVEL",
    "verdadeiro": "VERDADEIRO", "true": "VERDADEIRO", "comprovado": "VERDADEIRO",
    "certo": "VERDADEIRO", "correto": "VERDADEIRO", "fato": "VERDADEIRO",
    "fato verificado": "VERDADEIRO", "confirmado": "VERDADEIRO",
}

def _mapear_anterior(valor):
    if not isinstance(valor, str) or not valor.strip():
        return "NAO_CLASSIFICADO"
    return _MAPA_ANTERIOR.get(valor.strip().lower(), "OUTRO")

df["_avaliacao_antes"] = df["avaliacao_original"].apply(_mapear_anterior)

antes_dist  = df["_avaliacao_antes"].value_counts()
depois_dist = df["avaliacao_categoria"].value_counts()

sep = "=" * 62
print(sep)
print("  COMPARAÇÃO: MAPEAMENTO ANTERIOR vs. CORRIGIDO  (pré-dedup)")
print(sep)
print()
print(f"  {'Categoria':<24} {'Antes':>8} {'Depois':>8} {'Δ':>8}")
print("  " + "-" * 52)

todas_cats = sorted(set(antes_dist.index) | set(depois_dist.index))
for cat in todas_cats:
    a = antes_dist.get(cat, 0)
    d = depois_dist.get(cat, 0)
    delta = d - a
    sinal = "+" if delta > 0 else ""
    print(f"  {cat:<24} {a:>8} {d:>8} {sinal + str(delta):>8}")

print("  " + "-" * 52)

outro_antes  = antes_dist.get("OUTRO", 0)
outro_depois = depois_dist.get("OUTRO", 0)
recuperados  = outro_antes - outro_depois
print(f"\n  Registros OUTRO recuperados (bruto): {recuperados}")

# Detalhe: destino dos registros que saíram de OUTRO
mask = (df["_avaliacao_antes"] == "OUTRO") & (df["avaliacao_categoria"] != "OUTRO")
recuperados_df = df[mask].copy()

if len(recuperados_df) > 0:
    print("\n  Destino dos registros recuperados:")
    for cat, cnt in recuperados_df["avaliacao_categoria"].value_counts().items():
        print(f"    → {cat:<26} {cnt:>5} registros")

    print("\n  Top avaliacao_original recuperadas de OUTRO:")
    for orig, cnt in recuperados_df["avaliacao_original"].value_counts().head(20).items():
        cat = recuperados_df.loc[recuperados_df["avaliacao_original"] == orig, "avaliacao_categoria"].iloc[0]
        print(f"    {cnt:>4}x  {str(orig)[:60]:<62} → {cat}")

# Valores ainda como OUTRO
ainda_outro = df[df["avaliacao_categoria"] == "OUTRO"]["avaliacao_original"].value_counts()
print(f"\n{sep}")
print(f"  Valores ainda como OUTRO: {ainda_outro.sum()} registros / {len(ainda_outro)} únicos")
print(sep)
for orig, cnt in ainda_outro.head(25).items():
    print(f"  {cnt:>4}x  {str(orig)[:70]}")

print(f"\n  GFC_VERDADEIRO (pré-dedup): {depois_dist.get('VERDADEIRO', 0)}")
print(sep)

# Limpa coluna auxiliar
df.drop(columns=["_avaliacao_antes"], inplace=True)


  COMPARAÇÃO: MAPEAMENTO ANTERIOR vs. CORRIGIDO  (pré-dedup)

  Categoria                   Antes   Depois        Δ
  ----------------------------------------------------
  ENGANOSO                     2011     2535     +524
  FALSO                        6287     6871     +584
  FORA_DE_CONTEXTO              195      237      +42
  IMPRECISO                      12       12        0
  OUTRO                        1246       95    -1151
  VERDADEIRO                     67       68       +1
  ----------------------------------------------------

  Registros OUTRO recuperados (bruto): 1151

  Destino dos registros recuperados:
    → FALSO                        584 registros
    → ENGANOSO                     524 registros
    → FORA_DE_CONTEXTO              42 registros
    → VERDADEIRO                     1 registros

  Top avaliacao_original recuperadas de OUTRO:
     326x  Errado                                                         → FALSO
     270x  não_é_bem_assim               

## Remoção de duplicatas

In [19]:
# Dedup triplo: (1) texto+fonte+url, (2) texto normalizado, (3) url_origem
antes = len(df)

# Passo 1: chave composta
df = df.drop_duplicates(
    subset=["texto_principal", "fonte", "url_origem"],
    keep="first"
).copy()
n_dedup1 = antes - len(df)

# Passo 2: texto normalizado (independente de fonte)
df["_chave_texto"] = (
    df["texto_principal"]
    .str.lower().str.strip()
    .str.replace(r"\s+", " ", regex=True)
    .str.replace(r"[^\w\s]", "", regex=True)
)
antes2 = len(df)
df = df.drop_duplicates(subset=["_chave_texto"], keep="first").copy()
n_dedup2 = antes2 - len(df)

# Passo 3: url_origem (mesma checagem em diferentes queries)
antes3 = len(df)
mask_url_ok = df["url_origem"].fillna("").str.strip() != ""
df = pd.concat([
    df[mask_url_ok].drop_duplicates(subset=["url_origem"], keep="first"),
    df[~mask_url_ok],
], ignore_index=True)
n_dedup3 = antes3 - len(df)

df.drop(columns=["_chave_texto"], inplace=True)

print(f"Dedup passo 1 (texto+fonte+url)      : -{n_dedup1}")
print(f"Dedup passo 2 (texto normalizado)    : -{n_dedup2}")
print(f"Dedup passo 3 (url_origem)           : -{n_dedup3}")
print(f"Total removido                       : -{antes - len(df)}")
print(f"Registros únicos                     : {len(df)}")

Dedup passo 1 (texto+fonte+url)      : -4036
Dedup passo 2 (texto normalizado)    : -335
Dedup passo 3 (url_origem)           : -239
Total removido                       : -4610
Registros únicos                     : 5208


## Montagem do DataFrame curated

## Mapeamento de labels — `label`, `label_detalhe`, `status_curadoria`

| `avaliacao_categoria` | `label` | `label_detalhe` | `status_curadoria` |
|---|---|---|---|
| FALSO | 0 | FALSO | APROVADO_AUTO |
| ENGANOSO | 0 | ENGANOSO | APROVADO_AUTO |
| FORA_DE_CONTEXTO | 0 | FORA_DE_CONTEXTO | APROVADO_AUTO |
| IMPRECISO | 0 | ENGANOSO | APROVADO_AUTO |
| VERDADEIRO | 1 | GFC_VERDADEIRO | APROVADO_AUTO |
| NAO_VERIFICAVEL / NAO_CLASSIFICADO / OUTRO | — | PENDENTE_REVISAO | PENDENTE_REVISAO |

In [20]:
_LABEL_MAP = {
    "FALSO":           (0, "FALSO",            "APROVADO_AUTO"),
    "ENGANOSO":        (0, "ENGANOSO",          "APROVADO_AUTO"),
    "FORA_DE_CONTEXTO":(0, "FORA_DE_CONTEXTO",  "APROVADO_AUTO"),
    "IMPRECISO":       (0, "ENGANOSO",           "APROVADO_AUTO"),
    "VERDADEIRO":      (1, "GFC_VERDADEIRO",     "APROVADO_AUTO"),
}

def _mapear_label(cat: str):
    if cat in _LABEL_MAP:
        return _LABEL_MAP[cat]
    return (None, "PENDENTE_REVISAO", "PENDENTE_REVISAO")

_label_res = df["avaliacao_categoria"].apply(
    lambda c: pd.Series(_mapear_label(c), index=["label", "label_detalhe", "status_curadoria"])
)
df["label"]            = _label_res["label"]
df["label_detalhe"]    = _label_res["label_detalhe"]
df["status_curadoria"] = _label_res["status_curadoria"]

n_aprov  = (df["status_curadoria"] == "APROVADO_AUTO").sum()
n_pend   = (df["status_curadoria"] == "PENDENTE_REVISAO").sum()
n_label0 = (df["label"] == 0).sum()
n_label1 = (df["label"] == 1).sum()
print(f"APROVADO_AUTO  : {n_aprov}")
print(f"PENDENTE_REVISAO: {n_pend}")
print(f"label=0 (falsos/enganosos): {n_label0}")
print(f"label=1 (GFC_VERDADEIRO) : {n_label1}")
print(f"\nDistribuição label_detalhe:")
print(df["label_detalhe"].value_counts().to_string())


APROVADO_AUTO  : 5145
PENDENTE_REVISAO: 63
label=0 (falsos/enganosos): 5110
label=1 (GFC_VERDADEIRO) : 35

Distribuição label_detalhe:
label_detalhe
FALSO               3664
ENGANOSO            1301
FORA_DE_CONTEXTO     145
PENDENTE_REVISAO      63
GFC_VERDADEIRO        35


In [21]:
df["id_registro"]      = [str(uuid.uuid4()) for _ in range(len(df))]
df["pipeline"]          = "google_factcheck"
df["pipeline_origem"]   = "google_factcheck"
df["dataset_origem"]    = "CHECKAI_PROPRIO_GFC"
df["origem_qualidade"]  = "ROTULO_FORTE"
df["tipo_conteudo"]     = "AFIRMACAO_CHECADA"
df["data_curadoria"]    = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# rotulo_preliminar mantido para compatibilidade retroativa
df["rotulo_preliminar"] = df["avaliacao_categoria"].apply(
    lambda cat: "VERDADEIRO" if cat == "VERDADEIRO" else "FALSO"
)

# query_matched / tema_query / subtema_query: lê do raw se existir, senão vazio
for col in ["query_matched", "tema_query", "subtema_query"]:
    if col not in df.columns:
        df[col] = ""

# Colunas obrigatórias do schema curated
COLUNAS_OBRIGATORIAS = [
    "id_registro",
    "texto_principal",
    "label",
    "label_detalhe",
    "status_curadoria",
    "rotulo_preliminar",
    "pipeline",
    "pipeline_origem",
    "dataset_origem",
    "origem_qualidade",
    "fonte",
    "tipo_conteudo",
    "data_publicacao",
    "url_origem",
    "data_curadoria",
]

# Colunas de contexto específicas deste pipeline
COLUNAS_CONTEXTO = [
    "avaliacao_original",
    "avaliacao_categoria",
    "termo_busca",
    "query_matched",
    "tema_query",
    "subtema_query",
    "arquivo_raw_origem",
]

df_curated = df[COLUNAS_OBRIGATORIAS + COLUNAS_CONTEXTO].reset_index(drop=True)

print(f"Shape final do curated: {df_curated.shape}")
print(f"\nColunas: {list(df_curated.columns)}")
df_curated.head(3)

Shape final do curated: (5208, 22)

Colunas: ['id_registro', 'texto_principal', 'label', 'label_detalhe', 'status_curadoria', 'rotulo_preliminar', 'pipeline', 'pipeline_origem', 'dataset_origem', 'origem_qualidade', 'fonte', 'tipo_conteudo', 'data_publicacao', 'url_origem', 'data_curadoria', 'avaliacao_original', 'avaliacao_categoria', 'termo_busca', 'query_matched', 'tema_query', 'subtema_query', 'arquivo_raw_origem']


,id_registro,texto_principal,label,label_detalhe,status_curadoria,rotulo_preliminar,pipeline,pipeline_origem,dataset_origem,origem_qualidade,...,data_publicacao,url_origem,data_curadoria,avaliacao_original,avaliacao_categoria,termo_busca,query_matched,tema_query,subtema_query,arquivo_raw_origem
0,b0ae7418-65c1-45e9-bd7f-e76a68b02051,Lula perdeu todas as eleições com votação manu...,0.0,ENGANOSO,APROVADO_AUTO,FALSO,google_factcheck,google_factcheck,CHECKAI_PROPRIO_GFC,ROTULO_FORTE,...,2025-12-15,https://checamos.afp.com/doc.afp.com.88868T3,2026-05-30 21:49:09,Enganoso,ENGANOSO,urnas eletrônicas,NaN,NaN,NaN,google_factcheck_raw.csv
1,98a62d63-72cc-4c9b-8bea-e1e352cb19a1,Lula perdeu todas as eleições feitas com cédul...,0.0,FALSO,APROVADO_AUTO,FALSO,google_factcheck,google_factcheck,CHECKAI_PROPRIO_GFC,ROTULO_FORTE,...,2025-12-09,https://www.aosfatos.org/noticias/falso-que-lu...,2026-05-30 21:49:09,falso,FALSO,urnas eletrônicas,NaN,NaN,NaN,google_factcheck_raw.csv
2,c69d4152-e4d0-4574-a574-4624b5394641,No congresso americano todas as urnas foram ha...,0.0,FALSO,APROVADO_AUTO,FALSO,google_factcheck,google_factcheck,CHECKAI_PROPRIO_GFC,ROTULO_FORTE,...,2025-11-07,https://projetocomprova.com.br/publica%C3%A7%C...,2026-05-30 21:49:09,Falso,FALSO,urnas eletrônicas,NaN,NaN,NaN,google_factcheck_raw.csv


## Verificação de qualidade

In [22]:
print("=== Verificação de qualidade ===")
print(f"\nTotal de registros: {len(df_curated)}")
print(f"\nValores nulos por coluna:")
print(df_curated[COLUNAS_OBRIGATORIAS].isnull().sum())
print(f"\nDistribuição por fonte:")
print(df_curated["fonte"].value_counts().head(15))
print(f"\nDistribuição por avaliacao_categoria:")
print(df_curated["avaliacao_categoria"].value_counts())
print(f"\nRegistros com data_publicacao preenchida: {(df_curated['data_publicacao'] != '').sum()}")

=== Verificação de qualidade ===

Total de registros: 5208

Valores nulos por coluna:
id_registro           0
texto_principal       0
label                63
label_detalhe         0
status_curadoria      0
rotulo_preliminar     0
pipeline              0
pipeline_origem       0
dataset_origem        0
origem_qualidade      0
fonte                 0
tipo_conteudo         0
data_publicacao       0
url_origem            0
data_curadoria        0
dtype: int64

Distribuição por fonte:
fonte
AOS_FATOS              1266
ESTADÃO                1008
UOL_NOTÍCIAS            848
AFP_CHECAMOS            761
PROJETO_COMPROVA        367
BOATOS.ORG              291
OBSERVADOR              289
DESCONHECIDA            167
FOLHA_-_UOL             103
BOL_-_UOL                66
METRÓPOLES               18
AGÊNCIA_TATU              9
NEXO_JORNAL               6
CORREIO_BRAZILIENSE       5
ALETHEIA_FACT             2
Name: count, dtype: int64

Distribuição por avaliacao_categoria:
avaliacao_categoria
FALSO

## Por que esta correção foi necessária

### Contexto

A curadoria anterior usava um mapeamento conservador, projetado para evitar classificações
ambíguas. Isso era correto como postura inicial, mas resultou em ~500 registros classificados
como `OUTRO` que possuíam avaliações claras dos fact-checkers.

### Problemas identificados e corrigidos

**1. Mapa incompleto** — valores comuns sem mapeamento explícito:
- `Errado` (107 ocorrências) → agora `FALSO`
- `Insustentável` (37 ocorrências) → agora `FALSO`
- `Enganador` (15–17 ocorrências) → agora `ENGANOSO`
- `não_é_bem_assim` (91 ocorrências, Boatos.org) → agora `ENGANOSO`
- `Montagem` (9 ocorrências) → agora `FALSO`
- `Sátira` (7 ocorrências) → agora `FALSO`
- `Falta contexto` (4 ocorrências) → agora `FORA_DE_CONTEXTO`
- `confere` (ausente) → agora `VERDADEIRO`

**2. Encoding corrompido** — `não_é_bem_assim` era lido como `n\ufffd\ufffd_\ufffd_bem_assim`
em arquivos raw mais antigos. Corrigido com normalização unicode NFKC + transliteração ASCII.

**3. Ratings em forma de sentença** — fact-checkers como Comprova e AFP Checamos escrevem
o veredicto como frase completa: `"Falso: O vídeo mostra..."`. Nunca casavam com chaves
simples do mapa. Corrigido com fallback por prefixo (`falso:`, `enganoso:`, `comprovado:`, etc.).

### O que foi preservado

- **Nenhum arquivo raw foi alterado** — a camada raw permanece intacta.
- **Arquivos curated anteriores não foram sobrescritos** — novo arquivo tem timestamp próprio.
- **`dataset_final_treino_v1.csv` não foi alterado** — correção afeta apenas a camada curated.

### Valores que continuam como `OUTRO` propositalmente

`Contextualizando`, `Sem registro`, `Explica` e ratings em parágrafo sem prefixo identificável
não têm classificação unívoca e não devem entrar no treino sem revisão manual.


## Exportação para curated/

In [23]:
data_agora = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
caminho_saida = PASTA_CURATED / f"google_factcheck_curated_{data_agora}.csv"

df_curated.to_csv(caminho_saida, index=False, encoding="utf-8-sig")

print(f"Arquivo curated salvo em: {caminho_saida}")
print(f"Total de registros exportados: {len(df_curated)}")
print(f"Data e hora da curadoria: {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}")

Arquivo curated salvo em: ..\dados\pipeline_falso_google_factcheck\curated\google_factcheck_curated_2026-05-30_21-49-09.csv
Total de registros exportados: 5208
Data e hora da curadoria: 30/05/2026 21:49:09


In [24]:
# =====================================================================
# RELATÓRIO FINAL — métricas pós-exportação
# =====================================================================
sep = "=" * 64
print(sep)
print("  RELATÓRIO FINAL — CURADORIA GFC CORRIGIDA (v2)")
print(sep)

dist_final = df_curated["avaliacao_categoria"].value_counts()

print("\n  Distribuição final (pós-dedup):")
print(f"  {'Categoria':<24} {'Registros':>10} {'%':>8}")
print("  " + "-" * 46)
for cat, count in dist_final.items():
    pct = 100 * count / len(df_curated)
    print(f"  {cat:<24} {count:>10}  {pct:>6.1f}%")
print("  " + "-" * 46)
print(f"  {'TOTAL':<24} {len(df_curated):>10}")

n_verdadeiro = dist_final.get("VERDADEIRO", 0)
n_outro      = dist_final.get("OUTRO",      0)

print(f"\n{sep}")
print(f"  GFC_VERDADEIRO (label=1 de alta qualidade): {n_verdadeiro} registros")
print(sep)

verdadeiros_df = df_curated[df_curated["avaliacao_categoria"] == "VERDADEIRO"]
if len(verdadeiros_df) > 0:
    print("\n  Avaliações originais dos GFC_VERDADEIRO:")
    for orig, cnt in verdadeiros_df["avaliacao_original"].value_counts().items():
        print(f"    {cnt:>4}x  {orig}")
    print("\n  Fact-checkers dos GFC_VERDADEIRO:")
    for fonte, cnt in verdadeiros_df["fonte"].value_counts().items():
        print(f"    {cnt:>4}x  {fonte}")

print(f"\n{sep}")
print(f"  OUTRO residual (ambíguos — excluídos do treino): {n_outro} registros")
if n_outro > 0:
    outro_res = df_curated[df_curated["avaliacao_categoria"] == "OUTRO"]["avaliacao_original"].value_counts()
    print("  Principais valores restantes:")
    for orig, cnt in outro_res.head(15).items():
        print(f"    {cnt:>4}x  {str(orig)[:65]}")

# ── Distribuição por label/status ────────────────────────────────────────
if "label" in df_curated.columns:
    n_ap  = (df_curated["status_curadoria"] == "APROVADO_AUTO").sum()
    n_pe  = (df_curated["status_curadoria"] == "PENDENTE_REVISAO").sum()
    n_l0  = (df_curated["label"] == 0).sum()
    n_l1  = (df_curated["label"] == 1).sum()
    print(f"\n{sep}")
    print("  LABELS E STATUS")
    print(sep)
    print(f"  APROVADO_AUTO     : {n_ap}")
    print(f"  PENDENTE_REVISAO  : {n_pe}")
    print(f"  label=0 (falsos)  : {n_l0}")
    print(f"  label=1 (verds)   : {n_l1}")
    print(f"\n  Distribuição label_detalhe:")
    for ld, cnt in df_curated["label_detalhe"].value_counts().items():
        print(f"    {str(ld):<24} {cnt:>6}")

# ── Cobertura por query/tema ───────────────────────────────────────────────
if "query_matched" in df_curated.columns:
    df_com_q   = df_curated[df_curated["query_matched"].fillna("") != ""]
    df_sem_q   = df_curated[df_curated["query_matched"].fillna("") == ""]
    vc_query   = df_com_q["query_matched"].value_counts()
    vc_tema    = df_com_q["tema_query"].value_counts() if "tema_query" in df_com_q.columns else None

    # Queries definidas no pipeline autoral (referência)
    _QUERIES_REF = [
        "escala 6x1","fim da escala 6x1","jornada de trabalho","CLT","salário mínimo",
        "INSS","aposentadoria","FGTS","MEI","pejotização",
        "eleições 2026","urna eletrônica","fraude eleitoral","TSE","voto impresso",
        "biometria eleitoral","propaganda eleitoral","pesquisa eleitoral",
        "fake news eleitoral","deepfake eleições","inteligência artificial eleições",
        "STF","Alexandre de Moraes","Supremo Tribunal Federal","8 de janeiro",
        "Polícia Federal","impeachment","CPI","PEC","Congresso Nacional",
        "Câmara dos Deputados","Senado Federal","emendas parlamentares","orçamento secreto",
        "Pix","Banco Central","inflação","taxa Selic","preço da gasolina",
        "Bolsa Família","Auxílio Brasil","Minha Casa Minha Vida","reforma tributária",
        "imposto de renda","taxação da Shein","taxação de bets",
        "SUS","vacina","dengue","Covid","Enem","Fies","Prouni",
        "segurança pública","saidinha temporária","porte de armas",
        "Lula","Bolsonaro","Jair Bolsonaro","Michelle Bolsonaro","Eduardo Bolsonaro",
        "Tarcísio de Freitas","Guilherme Boulos","Pablo Marçal","Nikolas Ferreira",
        "André Janones","Arthur Lira","Rodrigo Pacheco","Sergio Moro",
        "MBL","Movimento Brasil Livre","PT","PL","PSOL","MDB","Novo","Centrão",
        "bolsonarismo","lulismo","lava jato",
        "kit gay","ideologia de gênero","comunismo no Brasil","Foro de São Paulo",
        "Venezuela","censura","liberdade de expressão","bloqueio de redes sociais",
        "vídeo manipulado","áudio falso","print falso","montagem política",
    ]
    matched_set  = set(df_com_q["query_matched"].unique())
    sem_result_q = sorted(set(_QUERIES_REF) - matched_set)

    print(f"\n{sep}")
    print("  COBERTURA POR QUERY")
    print(sep)
    print(f"  Registros com query matched : {len(df_com_q)} ({100*len(df_com_q)/max(len(df_curated),1):.1f}%)")
    print(f"  Registros sem query matched : {len(df_sem_q)}")
    print(f"  Queries com resultado       : {len(matched_set)} / {len(_QUERIES_REF)}")
    print(f"  Queries sem resultado       : {len(sem_result_q)}")
    print(f"\n  Top 30 queries com mais registros:")
    for q, cnt in vc_query.head(30).items():
        print(f"    {cnt:>5}x  {q}")
    if vc_tema is not None:
        print(f"\n  Cobertura por tema:")
        for t, cnt in vc_tema.items():
            print(f"    {cnt:>5}x  {t}")
    if sem_result_q:
        print(f"\n  Queries sem resultado ({len(sem_result_q)}):")
        for q in sem_result_q:
            print(f"    - {q}")

# ── Validação de Segurança dos Outputs ──────────────────────────────────
import re as _re2

_PADROES_SEC = [
    r"key=[A-Za-z0-9_\-]{10,}",
    r"AIza[A-Za-z0-9_\-]{10,}",
    r"GOOGLE_API_KEY",
    r"FACTCHECK_API_KEY",
]

def _scan_csv(caminho):
    achados = []
    try:
        with open(caminho, encoding="utf-8-sig", errors="replace") as _f:
            for n, linha in enumerate(_f, 1):
                for pat in _PADROES_SEC:
                    if _re2.search(pat, linha):
                        achados.append((n, pat))
                        break
    except Exception as exc:
        achados.append((0, str(exc)))
    return achados

_achados_cur = _scan_csv(caminho_saida)
_url_col_raw_present = "url_consulta" in df_raw.columns  # já deve estar False após sanitização
_url_col_cur_present = "url_consulta" in df_curated.columns

print(f"\n{sep}")
print("  VALIDAÇÃO DE SEGURANÇA DOS OUTPUTS")
print(sep)
print(f"  Padrões verificados        : key=... | AIza... | GOOGLE_API_KEY | FACTCHECK_API_KEY")
print(f"  url_consulta no raw        : {'PRESENTE (ALERTA)' if _url_col_raw_present else 'REMOVIDA (OK)'}")
print(f"  url_consulta no curated    : {'PRESENTE (ALERTA)' if _url_col_cur_present else 'AUSENTE (OK)'}")
if _achados_cur:
    print(f"  ALERTA: {len(_achados_cur)} linha(s) com padrão sensível em curated")
    for ln, pat in _achados_cur[:5]:
        print(f"    linha {ln}: [{pat}]")
else:
    print(f"  Ocorrências no curated     : 0 (OK)")
    print(f"  Arquivo seguro para commit : SIM")

print(f"\n{sep}")
print(f"  Arquivo salvo : {caminho_saida}")
print(f"  Data/hora     : {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}")
print(sep)


  RELATÓRIO FINAL — CURADORIA GFC CORRIGIDA (v2)

  Distribuição final (pós-dedup):
  Categoria                 Registros        %
  ----------------------------------------------
  FALSO                          3664    70.4%
  ENGANOSO                       1294    24.8%
  FORA_DE_CONTEXTO                145     2.8%
  OUTRO                            63     1.2%
  VERDADEIRO                       35     0.7%
  IMPRECISO                         7     0.1%
  ----------------------------------------------
  TOTAL                          5208

  GFC_VERDADEIRO (label=1 de alta qualidade): 35 registros

  Avaliações originais dos GFC_VERDADEIRO:
      17x  Verdadeiro
       9x  verdadeiro
       5x  Comprovado
       3x  Certo
       1x  COMPROVADO: É verdadeira a comparação de países que usam cloroquina no tratamento da covid-19 e outros que possuem protocolo para uso de cannabis medicinal.

  Fact-checkers dos GFC_VERDADEIRO:
      13x  UOL_NOTÍCIAS
       9x  AOS_FATOS
       5x  MET